In [1]:
import pickle
import pandas as pd
import numpy as np

# load saved model
with open('best_model.pkl', 'rb') as f:
    model = pickle.load(f)
print("model loaded!")

# load saved column names
with open('model_columns.pkl', 'rb') as f:
    model_columns = pickle.load(f)
print("columns loaded!")

# load and preprocess test data
test_df = pd.read_csv('test.csv')
test_df = test_df.fillna(0)
test_df = test_df.drop(columns=['User_ID', 'Product_ID'])
test_df = pd.get_dummies(test_df, columns=['Gender', 'Age', 'City_Category',
                                            'Stay_In_Current_City_Years'])

# align columns exactly as training
test_df = test_df.reindex(columns=model_columns, fill_value=0)

# predict
predictions = model.predict(test_df)

# save results
output = pd.DataFrame({'Predicted_Purchase': predictions})
output.to_csv('test_predictions.csv', index=False)
print("predictions saved!")
print(output.head(10))

model loaded!
columns loaded!
predictions saved!
   Predicted_Purchase
0        12530.381022
1        10687.479789
2         6098.664894
3         2627.797980
4         2778.255725
5        11687.158385
6        12529.185714
7        11349.905501
8        18906.547085
9         6098.664894


In [2]:
!pip install fastapi uvicorn nest-asyncio pyngrok


   ---------------- ----------------------- 2/5 [uvicorn]
   ---------------- ----------------------- 2/5 [uvicorn]
   ------------------------ --------------- 3/5 [starlette]
   -------------------------------- ------- 4/5 [fastapi]
   -------------------------------- ------- 4/5 [fastapi]
   ---------------------------------------- 5/5 [fastapi]



In [7]:
import pickle
import pandas as pd
import uvicorn
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

# ── fix jupyter event loop conflict ──────────────────
nest_asyncio.apply()

# ── load model and columns ────────────────────────────
with open('best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('model_columns.pkl', 'rb') as f:
    model_columns = pickle.load(f)

# ── define app ────────────────────────────────────────
app = FastAPI()

# ── input structure ───────────────────────────────────
class PurchaseInput(BaseModel):
    Gender: str
    Age: str
    Occupation: int
    City_Category: str
    Stay_In_Current_City_Years: str
    Marital_Status: int
    Product_Category_1: int
    Product_Category_2: float
    Product_Category_3: float

# ── routes ────────────────────────────────────────────
@app.get('/')
def home():
    return {'message': 'API is running!'}

@app.post('/predict')
def predict(data: PurchaseInput):
    input_dict = dict(data)
    input_df = pd.DataFrame([input_dict])
    input_df = pd.get_dummies(input_df, columns=['Gender', 'Age', 'City_Category',
                                                   'Stay_In_Current_City_Years'])
    input_df = input_df.reindex(columns=model_columns, fill_value=0)
    prediction = model.predict(input_df)[0]
    return {
        'predicted_purchase': round(float(prediction), 2),
        'model_used': str(type(model).__name__)
    }

# ── run ───────────────────────────────────────────────
uvicorn.run(app, host='0.0.0.0', port=8000)

# ❌ Your API was not strictly defining what input format it expects.

# FastAPI solves this using Pydantic models.

RuntimeError: asyncio.run() cannot be called from a running event loop

In [8]:
import requests

sample = {
    "Gender": "M",
    "Age": "26-35",
    "Occupation": 4,
    "City_Category": "A",
    "Stay_In_Current_City_Years": "1",
    "Marital_Status": 0,
    "Product_Category_1": 5,
    "Product_Category_2": 0.0,
    "Product_Category_3": 0.0
}

response = requests.post("http://127.0.0.1:8000/predict", json=sample)
print(response.json())

{'predicted_purchase': 5963.4, 'model_used': 'DecisionTreeRegressor'}


In [9]:
app_code = '''
import pickle
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

with open("best_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("model_columns.pkl", "rb") as f:
    model_columns = pickle.load(f)

app = FastAPI()

class PurchaseInput(BaseModel):
    Gender: str
    Age: str
    Occupation: int
    City_Category: str
    Stay_In_Current_City_Years: str
    Marital_Status: int
    Product_Category_1: int
    Product_Category_2: float
    Product_Category_3: float

@app.get("/")
def home():
    return {"message": "API is running!"}

@app.post("/predict")
def predict(data: PurchaseInput):
    input_dict = dict(data)
    input_df = pd.DataFrame([input_dict])
    input_df = pd.get_dummies(input_df, columns=["Gender", "Age", "City_Category",
                                                   "Stay_In_Current_City_Years"])
    input_df = input_df.reindex(columns=model_columns, fill_value=0)
    prediction = model.predict(input_df)[0]
    return {
        "predicted_purchase": round(float(prediction), 2),
        "model_used": str(type(model).__name__)
    }
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [5]:
import os
print(os.listdir(os.getcwd()))

['.ipynb_checkpoints', 'app.py', 'best_model.pkl', 'models_practice.ipynb', 'model_columns.pkl', 'model_scores.json', 'predictions.csv', 'predictions.ipynb', 'test.csv', 'test_predictions.csv', 'train.csv', 'training.ipynb', '__pycache__']


In [6]:
import requests

sample = {
    "Gender": "M",
    "Age": "26-35",
    "Occupation": 4,
    "City_Category": "A",
    "Stay_In_Current_City_Years": "1",
    "Marital_Status": 0,
    "Product_Category_1": 5,
    "Product_Category_2": 0.0,
    "Product_Category_3": 0.0
}

response = requests.post("http://127.0.0.1:8000/predict", json=sample)
print(response.json())

{'predicted_purchase': 5963.4, 'model_used': 'DecisionTreeRegressor'}
